# Proyecto Integrador — Hito 1: Construcción del Dataset Analítico

## Enfoque seleccionado
Opción 1 — Enfoque de Ventas.

Integramos información de pedidos, detalle de pedidos, productos y campañas de marketing para analizar ventas, cantidades, descuentos, ganancias, productos, categorías y su relación con las estrategias de marketing.

## Integrantes
- Facundo Vazquez
- Matias Alejandro Gomez
- Federico Ezequiel Castillo
- José Luis Rodríguez
- Fidel Fernández Fontenla


# Fase 1: Exploración de la base de datos

En esta fase conectamos Colab con la base de datos y revisamos qué tablas contiene.
El objetivo es entender la estructura general antes de decidir qué información usar.


In [ ]:
#Carga de librerías y BD
import sqlite3
import pandas as pd

%reload_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

%sql sqlite:///34_SuperTienda_Espanol.db


In [ ]:
#Identificamos las tablas disponibles
%%sql
SELECT name FROM sqlite_master WHERE type='table';


La base de datos contiene 8 tablas: Clientes, Productos, Geografia, Cliente_Ubicacion, Campanias, Pedidos, Detalle_Pedido y Soporte.

A continuación exploramos las columnas de las tablas relevantes para nuestro enfoque de Ventas.


In [ ]:
#Identifico la tabla Pedidos
%%sql
SELECT * FROM Pedidos LIMIT 3;


In [ ]:
#Identifico la tabla Detalle_Pedido
%%sql
SELECT * FROM Detalle_Pedido LIMIT 3;


In [ ]:
#Identifico la tabla Productos
%%sql
SELECT * FROM Productos LIMIT 3;


In [ ]:
#Identifico la tabla Campanias
%%sql
SELECT * FROM Campanias LIMIT 3;


## Decisión de tablas

Para el enfoque de ventas seleccionamos 4 de las 8 tablas disponibles:

- **Pedidos** — Contiene los pedidos con fecha, modo de envío y estado.
- **Detalle_Pedido** — Contiene el detalle económico de cada pedido: ventas, cantidad, descuento y ganancia.
- **Productos** — Aporta categoría, subcategoría, marca, nombre, precio de lista y costo unitario del producto.
- **Campanias** — Permite vincular las ventas con las estrategias de marketing (nombre de campaña y canal), para evaluar el efecto de las acciones comerciales sobre las ventas.

Descartamos las tablas Clientes, Geografia, Cliente_Ubicacion y Soporte porque exceden el alcance de un análisis centrado en el desempeño de las transacciones comerciales en este primer hito.


# Fase 2: Extracción de datos (SQL y Pandas)

Construimos una consulta SQL que integra las 4 tablas seleccionadas mediante `JOIN` y la convertimos en un DataFrame de Pandas. Cada fila del resultado representa un producto vendido dentro de un pedido, con su campaña de marketing asociada (si la tuviera).


In [ ]:
%%sql resultado <<

SELECT
    p.ID_Pedido,
    p.Fecha_Pedido,
    p.Fecha_Envio,
    p.Modo_Envio,
    p.Estado_Pedido,

    dp.ID_Detalle,
    dp.ID_Producto,
    dp.Cantidad,
    dp.Descuento,
    dp.Ventas,
    dp.Ganancia,

    pr.Nombre_Producto,
    pr.Categoria,
    pr.Subcategoria,
    pr.Marca,
    pr.Precio_Lista,
    pr.Costo_Unitario,

    c.Nombre_Campania,
    c.Canal_Marketing

FROM Pedidos p
JOIN Detalle_Pedido dp
    ON p.ID_Pedido = dp.ID_Pedido
JOIN Productos pr
    ON dp.ID_Producto = pr.ID_Producto
LEFT JOIN Campanias c
    ON p.ID_Campania = c.ID_Campania;


In [ ]:
# Convertimos el resultado SQL en un DataFrame de Pandas
df = resultado.DataFrame()
df.head()


## Decisión de columnas

Seleccionamos columnas que permiten analizar ventas desde distintos ángulos:

- **¿Cuándo se vendió?**: `Fecha_Pedido`, `Fecha_Envio`
- **¿Cómo fue el pedido?**: `Modo_Envio`, `Estado_Pedido`
- **¿Qué se vendió?**: `Nombre_Producto`, `Categoria`, `Subcategoria`, `Marca`
- **¿Cuánto se vendió?**: `Cantidad`, `Ventas`, `Descuento`, `Ganancia`
- **Información económica**: `Precio_Lista`, `Costo_Unitario`
- **Información de marketing**: `Nombre_Campania`, `Canal_Marketing`


## 3. Limpieza y transformación (Python – Pandas)

### 3.1 Inspección inicial
Revisamos tipos de datos, valores nulos y estadísticas generales.


In [ ]:
#Tipos de datos
df.info()


In [ ]:
#Valores nulos
df.isna().sum()


In [ ]:
#Verificamos duplicados
df.duplicated().sum()


In [ ]:
#Estadisticas generales
df.describe()


### 3.2 Corrección de fechas

Convertimos las columnas de fecha a formato `datetime` para poder hacer análisis temporales (estacionalidad de ventas, tiempos logísticos, etc.).


In [ ]:
df['Fecha_Pedido'] = pd.to_datetime(df['Fecha_Pedido'], errors='coerce')
df['Fecha_Envio'] = pd.to_datetime(df['Fecha_Envio'], errors='coerce')

print(df['Fecha_Pedido'].dtype)
print(df['Fecha_Envio'].dtype)


### 3.3 Corrección de columnas numéricas

Nos aseguramos de que todas las columnas que representan importes, costos o cantidades estén correctamente asignadas a tipos numéricos.


In [ ]:
columnas_numericas = ['Cantidad', 'Descuento', 'Ventas', 'Ganancia', 'Precio_Lista', 'Costo_Unitario']

for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.info()


### 3.4 Estandarización de texto

Unificamos las columnas categóricas en minúsculas y sin espacios sobrantes, para evitar duplicados invisibles en las agrupaciones (por ejemplo, diferencias por mayúsculas o espacios extra).


In [ ]:
columnas_texto = ['Modo_Envio', 'Estado_Pedido', 'Nombre_Producto', 'Categoria',
                  'Subcategoria', 'Marca', 'Nombre_Campania', 'Canal_Marketing']

for col in columnas_texto:
    df[col] = df[col].fillna('sin dato').astype(str).str.lower().str.strip()

print(df['Categoria'].unique())
print(df['Modo_Envio'].unique())


### 3.5 Evaluación de calidad general del dataset

Revisamos estadísticas descriptivas para detectar posibles inconsistencias, errores o anomalías que puedan afectar la interpretación de los datos.


In [ ]:
df.describe()


**Hallazgo:** se detectaron valores negativos en la columna `Ganancia`, lo que indica que algunas transacciones generaron pérdidas, posiblemente por descuentos elevados. No eliminamos estos registros ya que representan información real del negocio y son relevantes para el análisis de rentabilidad.

### 3.6 Creación de variables calculadas (Feature Engineering)

Para enriquecer el análisis, creamos las siguientes variables:

1. **`Ingreso_Neto`**: valor real de cada venta luego de aplicar el descuento correspondiente. Fórmula: `Ingreso_Neto = Ventas * (1 - Descuento)`.
2. **`Venta_Promedio_Unidad`**: ticket unitario efectivo de cada transacción.
3. **`Mes_Pedido`**: mes calendario de la compra, útil para análisis de estacionalidad.
4. **`Dias_Envio`**: tiempo logístico (lead time) desde el pedido hasta el despacho.


In [ ]:
# 1. Ingreso neto real de la venta
df['Ingreso_Neto'] = df['Ventas'] * (1 - df['Descuento'])

# 2. Venta promedio por unidad de producto
df['Venta_Promedio_Unidad'] = (df['Ventas'] / df['Cantidad']).round(2)

# 3. Mes calendario de la compra
df['Mes_Pedido'] = df['Fecha_Pedido'].dt.month

# 4. Días requeridos para el envío del pedido
df['Dias_Envio'] = (df['Fecha_Envio'] - df['Fecha_Pedido']).dt.days

df[['Ingreso_Neto', 'Venta_Promedio_Unidad', 'Mes_Pedido', 'Dias_Envio']].describe()


### 3.7 Verificación final

Confirmamos el tamaño del dataset, sus columnas y la ausencia de valores nulos.


In [ ]:
print(f'Total filas: {len(df)}')
print(f'Total columnas: {len(df.columns)}')
print(f'Columnas: {df.columns.tolist()}')

print('\nNulos por columna:')
print(df.isna().sum())

df.head(10)


# Fase 4: Exportación del dataset

Exportamos el dataset limpio en formato CSV para utilizarlo en el siguiente módulo.


In [ ]:
nombre_archivo = 'Hito_1_Ventas_SuperTienda.csv'

# Usamos sep=';' y encoding='utf-8-sig' porque Excel en español interpreta
# la coma como separador decimal, no de columnas. Con sep=',' (default),
# Excel abre el archivo con todo el contenido mezclado en una sola columna.
df.to_csv(nombre_archivo, index=False, sep=';', encoding='utf-8-sig', float_format='%.2f')

print(f'Dataset exportado correctamente como {nombre_archivo}')


In [ ]:
from google.colab import files

files.download(nombre_archivo)


# Reflexión analítica

## ¿Qué tablas decidimos utilizar y por qué?

Utilizamos `Pedidos`, `Detalle_Pedido`, `Productos` y `Campanias` porque son las tablas relacionadas con el enfoque de ventas y su contexto comercial:

- **`Pedidos`**: aporta información general del pedido (fecha, envío, estado).
- **`Detalle_Pedido`**: contiene las métricas principales de venta (cantidad, ventas, descuento, ganancia).
- **`Productos`**: permite analizar las ventas por producto, categoría, subcategoría y marca.
- **`Campanias`**: permite vincular la facturación con los canales de marketing aplicados, facilitando el análisis de retorno de las acciones publicitarias.

Descartamos `Clientes`, `Geografia`, `Cliente_Ubicacion` y `Soporte` porque exceden el alcance de un análisis centrado en el desempeño de las transacciones comerciales en este primer hito.

## ¿Qué columnas seleccionamos y cuáles descartamos?

**Seleccionadas** (y presentes en la consulta SQL real): `ID_Pedido`, `Fecha_Pedido`, `Fecha_Envio`, `Modo_Envio`, `Estado_Pedido`, `ID_Detalle`, `ID_Producto`, `Cantidad`, `Descuento`, `Ventas`, `Ganancia`, `Nombre_Producto`, `Categoria`, `Subcategoria`, `Marca`, `Precio_Lista`, `Costo_Unitario`, `Nombre_Campania`, `Canal_Marketing`.

**Descartadas**: `ID_Cliente`, datos de ubicación geográfica y de soporte, por no ser necesarias para este análisis inicial de ventas.

## ¿Qué tipo de análisis permitirá este dataset en el siguiente módulo?

El dataset permitirá construir:

1. **Análisis de rentabilidad y eficiencia**: comparando `Ventas` vs `Ingreso_Neto` y `Ganancia`, identificando productos o categorías de bajo margen o transacciones que operan a pérdida por descuentos elevados.
2. **Análisis de desempeño comercial y marketing**: determinando qué categorías, marcas y campañas lideran la facturación.
3. **Análisis logístico y temporal**: mapeando la estacionalidad de las compras (`Mes_Pedido`) y los tiempos de entrega (`Dias_Envio`) según el modo de envío.
4. **Ranking de productos**: por cantidad y por monto de ventas.
